# M3N-VC Dataset Exploration

## Multi-Modality Multi-Node Vehicle Classification Dataset

This notebook provides a comprehensive exploration of the M3N-VC dataset, which contains synchronized microphone and geophone recordings of four different vehicles captured in multiple real-world scenes using spatially distributed sensor networks.

### Dataset Overview:
- **Modalities**: Microphone (1.6 kHz), Geophone (200 Hz), GPS (1 Hz)
- **Scenes**: 6 unique environments (h08, h24, s31, a06, i29, i22)
- **Vehicles**: Mazda CX-30 (cx30), Mercedes-Benz GLE 350 (gle350), Ford Mustang (mustang), Mazda MX-5 (miata)
- **Nodes**: 6-8 sensor nodes per scene
- **Total Duration**: ~18 hours of recordings

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

# For parquet files
try:
    import polars as pl
    POLARS_AVAILABLE = True
except ImportError:
    POLARS_AVAILABLE = False
    print("Polars not available, using pandas only")

warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("Libraries imported successfully!")

## 2. Dataset Overview and Summary

Let's start by loading the dataset summary to understand the high-level characteristics of each scene.

In [ ]:
# Load dataset summary
DATA_ROOT = Path('/home/lvc_toolkit/datasets/M3NVC')
summary_df = pd.read_csv(DATA_ROOT / 'm3n_dataset_summary.csv')
print("Dataset Summary:")
print(f"Total scenes: {len(summary_df)}")
print(f"Total hours: {summary_df['total_hours'].sum():.2f}")
print(f"Train hours: {summary_df['train_hours'].sum():.2f}")
print(f"Test hours: {summary_df['test_hours'].sum():.2f}")
print("\n")
summary_df

In [ ]:
# Visualize dataset summary
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Scene durations
ax1 = axes[0, 0]
summary_df[['scene', 'train_hours', 'test_hours']].set_index('scene').plot(
    kind='bar', stacked=True, ax=ax1, color=['#3498db', '#e74c3c']
)
ax1.set_title('Train/Test Split by Scene', fontsize=14, fontweight='bold')
ax1.set_ylabel('Hours')
ax1.set_xlabel('Scene')
ax1.legend(['Train', 'Test'])
ax1.grid(alpha=0.3)

# Number of nodes per scene
ax2 = axes[0, 1]
summary_df.plot(x='scene', y='nodes', kind='bar', ax=ax2, color='#2ecc71', legend=False)
ax2.set_title('Number of Sensor Nodes per Scene', fontsize=14, fontweight='bold')
ax2.set_ylabel('Number of Nodes')
ax2.set_xlabel('Scene')
ax2.grid(alpha=0.3)

# Terrain distribution
ax3 = axes[1, 0]
terrain_counts = summary_df['terrain'].value_counts()
ax3.pie(terrain_counts.values, labels=terrain_counts.index, autopct='%1.1f%%', 
        colors=['#3498db', '#e74c3c', '#f39c12', '#9b59b6'])
ax3.set_title('Terrain Distribution', fontsize=14, fontweight='bold')

# Weather distribution
ax4 = axes[1, 1]
weather_counts = summary_df['weather'].value_counts()
ax4.pie(weather_counts.values, labels=weather_counts.index, autopct='%1.1f%%',
        colors=['#f39c12', '#3498db', '#e74c3c'])
ax4.set_title('Weather Conditions Distribution', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Explore Scene Structure

Let's examine a specific scene (s31) to understand the data structure in detail.

In [ ]:
# Define scenes to explore
scenes = ['a06', 'h08', 'h24', 'i22', 'i29', 's31']
sample_scene = 's31'  # We'll use s31 as our primary example

print(f"Exploring scene: {sample_scene}")
print(f"Available scenes: {scenes}")

### 3.1 Load Run Metadata

In [ ]:
# Load run metadata for sample scene
run_ids_path = f'{sample_scene}/{sample_scene}/run_ids.parquet'

if POLARS_AVAILABLE:
    run_ids = pl.read_parquet(run_ids_path)
    print("Using Polars to read parquet file")
    run_ids_df = run_ids.to_pandas()
else:
    run_ids_df = pd.read_parquet(run_ids_path)
    print("Using Pandas to read parquet file")

print(f"\nRun metadata for scene {sample_scene}:")
print(f"Total runs: {len(run_ids_df)}")
print(f"Columns: {list(run_ids_df.columns)}")
print("\n")
run_ids_df

In [ ]:
# Analyze run durations and splits
if 'length' in run_ids_df.columns:
    # Convert duration to minutes for easier interpretation
    run_ids_df['duration_minutes'] = pd.to_timedelta(run_ids_df['length']).dt.total_seconds() / 60

print("Run Statistics:")
print(f"Vehicle labels: {run_ids_df['label'].unique()}")
print(f"Train runs: {(run_ids_df['set'] == 'train').sum()}")
print(f"Test runs: {(run_ids_df['set'] == 'test').sum()}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Run durations by vehicle
ax1 = axes[0]
for label in run_ids_df['label'].unique():
    mask = run_ids_df['label'] == label
    ax1.bar(run_ids_df[mask]['run_id'], run_ids_df[mask]['duration_minutes'], 
            label=label, alpha=0.7)
ax1.set_xlabel('Run ID')
ax1.set_ylabel('Duration (minutes)')
ax1.set_title('Run Durations by Vehicle Type')
ax1.legend()
ax1.grid(alpha=0.3)

# Train/Test distribution by vehicle
ax2 = axes[1]
train_test_counts = run_ids_df.groupby(['label', 'set']).size().unstack(fill_value=0)
train_test_counts.plot(kind='bar', ax=ax2, color=['#3498db', '#e74c3c'])
ax2.set_title('Train/Test Split by Vehicle Type')
ax2.set_ylabel('Number of Runs')
ax2.set_xlabel('Vehicle')
ax2.legend(['Train', 'Test'])
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 3.2 Sensor Node Locations

In [ ]:
# Load sensor locations
sensor_loc_path = f'{sample_scene}/{sample_scene}/sensor_location.parquet'

if POLARS_AVAILABLE:
    sensor_loc = pl.read_parquet(sensor_loc_path)
    sensor_loc_df = sensor_loc.to_pandas()
else:
    sensor_loc_df = pd.read_parquet(sensor_loc_path)

print(f"Sensor locations for scene {sample_scene}:")
print(f"Number of sensors: {len(sensor_loc_df)}")
print(f"Columns: {list(sensor_loc_df.columns)}")
print("\n")
sensor_loc_df

In [ ]:
# Visualize sensor node locations (if lat/lon columns exist)
if 'latitude' in sensor_loc_df.columns and 'longitude' in sensor_loc_df.columns:
    plt.figure(figsize=(12, 8))
    plt.scatter(sensor_loc_df['longitude'], sensor_loc_df['latitude'], 
                s=200, c='red', marker='o', edgecolors='black', linewidths=2, alpha=0.7)
    
    # Add node labels
    for idx, row in sensor_loc_df.iterrows():
        plt.annotate(f"Node {row.get('node_id', idx)}", 
                    (row['longitude'], row['latitude']),
                    xytext=(5, 5), textcoords='offset points', fontsize=10)
    
    plt.xlabel('Longitude', fontsize=12)
    plt.ylabel('Latitude', fontsize=12)
    plt.title(f'Sensor Node Locations - Scene {sample_scene}', fontsize=14, fontweight='bold')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Latitude/Longitude columns not found in sensor location data")
    print("Available columns:", sensor_loc_df.columns.tolist())

## 4. Explore Sensor Time Series Data

Now let's load and visualize actual sensor readings (microphone and geophone data).

### 4.1 Load Microphone Data

In [ ]:
# Load a sample microphone recording
sample_run = 0
sample_node = 1
mic_path = f'{sample_scene}/{sample_scene}/run{sample_run}_rs{sample_node}_mic.parquet'

try:
    if POLARS_AVAILABLE:
        mic_data = pl.read_parquet(mic_path)
        mic_df = mic_data.to_pandas()
    else:
        mic_df = pd.read_parquet(mic_path)
    
    # Convert timestamp to datetime if it exists
    if 'timestamp' in mic_df.columns:
        mic_df['datetime'] = pd.to_datetime(mic_df['timestamp'], unit='s')
    
    # Get signal column name
    signal_col = 'samples' if 'samples' in mic_df.columns else mic_df.columns[-1]
    
    print(f"📊 Microphone Data - Run {sample_run}, Node {sample_node}")
    print("=" * 70)
    print(f"Total Samples: {len(mic_df):,}")
    print(f"Duration: {len(mic_df) / 1600:.2f} seconds ({len(mic_df) / 1600 / 60:.2f} minutes)")
    print(f"Sampling Rate: 1,600 Hz")
    print(f"\nSignal Statistics:")
    print(f"  Mean: {mic_df[signal_col].mean():.2f}")
    print(f"  Std Dev: {mic_df[signal_col].std():.2f}")
    print(f"  Min: {mic_df[signal_col].min():.2f}")
    print(f"  Max: {mic_df[signal_col].max():.2f}")
    
    if 'datetime' in mic_df.columns:
        print(f"\nRecording Time:")
        print(f"  Start: {mic_df['datetime'].iloc[0]}")
        print(f"  End: {mic_df['datetime'].iloc[-1]}")
    
    print("\n" + "=" * 70)
    print("\nFirst 10 samples:")
    if 'datetime' in mic_df.columns:
        display_df = mic_df[['datetime', signal_col]].head(10)
    else:
        display_df = mic_df.head(10)
    display(display_df)
    
except FileNotFoundError:
    print(f"❌ File not found: {mic_path}")
except Exception as e:
    print(f"❌ Error loading microphone data: {e}")

In [ ]:
# Visualize microphone signal
if 'mic_df' in locals() and not mic_df.empty:
    # Determine the signal column
    signal_col = 'samples' if 'samples' in mic_df.columns else None
    if signal_col is None:
        for col in ['value', 'signal', 'amplitude', 'data', 'mic']:
            if col in mic_df.columns:
                signal_col = col
                break
    
    if signal_col is None and len(mic_df.columns) > 0:
        # Take the first numeric column
        numeric_cols = mic_df.select_dtypes(include=[np.number]).columns
        if len(numeric_cols) > 0:
            signal_col = numeric_cols[0]
    
    if signal_col:
        # Plot first 10 seconds of data
        sampling_rate = 1600  # Hz
        duration_to_plot = 10  # seconds
        samples_to_plot = min(sampling_rate * duration_to_plot, len(mic_df))
        time_axis = np.arange(samples_to_plot) / sampling_rate
        
        fig, axes = plt.subplots(2, 1, figsize=(16, 10))
        
        # Time domain plot
        ax1 = axes[0]
        ax1.plot(time_axis, mic_df[signal_col].iloc[:samples_to_plot], 
                linewidth=0.7, color='#2E86AB', alpha=0.8)
        ax1.set_xlabel('Time (seconds)', fontsize=12)
        ax1.set_ylabel('Amplitude', fontsize=12)
        ax1.set_title(f'🎤 Microphone Signal (Time Domain) - Run {sample_run}, Node {sample_node}\nFirst {duration_to_plot} seconds | Sampling Rate: {sampling_rate} Hz', 
                     fontsize=14, fontweight='bold', pad=15)
        ax1.grid(alpha=0.3, linestyle='--')
        ax1.set_facecolor('#f8f9fa')
        
        # Add statistics box
        signal_data = mic_df[signal_col].iloc[:samples_to_plot]
        stats_text = f'Mean: {signal_data.mean():.1f}\nStd: {signal_data.std():.1f}\nPeak: {signal_data.abs().max():.1f}'
        ax1.text(0.02, 0.98, stats_text, transform=ax1.transAxes, 
                fontsize=10, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))
        
        # Frequency domain plot (FFT)
        ax2 = axes[1]
        signal_segment = mic_df[signal_col].iloc[:samples_to_plot].values
        fft_vals = np.fft.rfft(signal_segment)
        fft_freq = np.fft.rfftfreq(len(signal_segment), 1/sampling_rate)
        fft_magnitude = np.abs(fft_vals)
        
        # Convert to dB scale for better visualization
        fft_magnitude_db = 20 * np.log10(fft_magnitude + 1e-10)
        
        ax2.plot(fft_freq, fft_magnitude_db, linewidth=0.8, color='#A23B72', alpha=0.8)
        ax2.set_xlabel('Frequency (Hz)', fontsize=12)
        ax2.set_ylabel('Magnitude (dB)', fontsize=12)
        ax2.set_title('🔊 Frequency Spectrum (FFT)', fontsize=14, fontweight='bold', pad=15)
        ax2.set_xlim([0, 800])  # Show up to Nyquist frequency
        ax2.grid(alpha=0.3, linestyle='--')
        ax2.set_facecolor('#f8f9fa')
        
        # Mark dominant frequencies
        peak_idx = np.argmax(fft_magnitude[1:]) + 1  # Skip DC component
        peak_freq = fft_freq[peak_idx]
        ax2.axvline(peak_freq, color='red', linestyle=':', linewidth=2, alpha=0.6, label=f'Peak: {peak_freq:.1f} Hz')
        ax2.legend(loc='upper right')
        
        plt.tight_layout()
        plt.show()
    else:
        print("❌ Could not determine signal column in microphone data")
else:
    print("⚠️  No microphone data loaded")

### 4.2 Load Geophone Data

In [ ]:
# Load geophone data from the same run and node
geo_path = f'{sample_scene}/{sample_scene}/run{sample_run}_rs{sample_node}_geo.parquet'

try:
    if POLARS_AVAILABLE:
        geo_data = pl.read_parquet(geo_path)
        geo_df = geo_data.to_pandas()
    else:
        geo_df = pd.read_parquet(geo_path)
    
    # Convert timestamp to datetime if it exists
    if 'timestamp' in geo_df.columns:
        geo_df['datetime'] = pd.to_datetime(geo_df['timestamp'], unit='s')
    
    # Get signal column name
    signal_col = 'samples' if 'samples' in geo_df.columns else geo_df.columns[-1]
    
    print(f"📊 Geophone Data - Run {sample_run}, Node {sample_node}")
    print("=" * 70)
    print(f"Total Samples: {len(geo_df):,}")
    print(f"Duration: {len(geo_df) / 200:.2f} seconds ({len(geo_df) / 200 / 60:.2f} minutes)")
    print(f"Sampling Rate: 200 Hz")
    print(f"\nSignal Statistics:")
    print(f"  Mean: {geo_df[signal_col].mean():.2f}")
    print(f"  Std Dev: {geo_df[signal_col].std():.2f}")
    print(f"  Min: {geo_df[signal_col].min():.2f}")
    print(f"  Max: {geo_df[signal_col].max():.2f}")
    
    if 'datetime' in geo_df.columns:
        print(f"\nRecording Time:")
        print(f"  Start: {geo_df['datetime'].iloc[0]}")
        print(f"  End: {geo_df['datetime'].iloc[-1]}")
    
    print("\n" + "=" * 70)
    print("\nFirst 10 samples:")
    if 'datetime' in geo_df.columns:
        display_df = geo_df[['datetime', signal_col]].head(10)
    else:
        display_df = geo_df.head(10)
    display(display_df)
    
except FileNotFoundError:
    print(f"❌ File not found: {geo_path}")
except Exception as e:
    print(f"❌ Error loading geophone data: {e}")

In [ ]:
# Visualize geophone signal
if 'geo_df' in locals() and not geo_df.empty:
    # Determine the signal column
    signal_col = 'samples' if 'samples' in geo_df.columns else None
    if signal_col is None:
        for col in ['value', 'signal', 'amplitude', 'data', 'geo', 'x', 'y', 'z']:
            if col in geo_df.columns:
                signal_col = col
                break
    
    if signal_col is None and len(geo_df.columns) > 0:
        numeric_cols = geo_df.select_dtypes(include=[np.number]).columns
        if len(numeric_cols) > 0:
            signal_col = numeric_cols[0]
    
    if signal_col:
        # Plot first 10 seconds of data
        sampling_rate = 200  # Hz
        duration_to_plot = 10  # seconds
        samples_to_plot = min(sampling_rate * duration_to_plot, len(geo_df))
        time_axis = np.arange(samples_to_plot) / sampling_rate
        
        fig, axes = plt.subplots(2, 1, figsize=(16, 10))
        
        # Time domain plot
        ax1 = axes[0]
        ax1.plot(time_axis, geo_df[signal_col].iloc[:samples_to_plot], 
                linewidth=0.9, color='#18A558', alpha=0.8)
        ax1.set_xlabel('Time (seconds)', fontsize=12)
        ax1.set_ylabel('Amplitude', fontsize=12)
        ax1.set_title(f'🌍 Geophone Signal (Time Domain) - Run {sample_run}, Node {sample_node}\nFirst {duration_to_plot} seconds | Sampling Rate: {sampling_rate} Hz', 
                     fontsize=14, fontweight='bold', pad=15)
        ax1.grid(alpha=0.3, linestyle='--')
        ax1.set_facecolor('#f8f9fa')
        
        # Add statistics box
        signal_data = geo_df[signal_col].iloc[:samples_to_plot]
        stats_text = f'Mean: {signal_data.mean():.1f}\nStd: {signal_data.std():.1f}\nPeak: {signal_data.abs().max():.1f}'
        ax1.text(0.02, 0.98, stats_text, transform=ax1.transAxes, 
                fontsize=10, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))
        
        # Frequency domain plot (FFT)
        ax2 = axes[1]
        signal_segment = geo_df[signal_col].iloc[:samples_to_plot].values
        fft_vals = np.fft.rfft(signal_segment)
        fft_freq = np.fft.rfftfreq(len(signal_segment), 1/sampling_rate)
        fft_magnitude = np.abs(fft_vals)
        
        # Convert to dB scale for better visualization
        fft_magnitude_db = 20 * np.log10(fft_magnitude + 1e-10)
        
        ax2.plot(fft_freq, fft_magnitude_db, linewidth=0.9, color='#0D7377', alpha=0.8)
        ax2.set_xlabel('Frequency (Hz)', fontsize=12)
        ax2.set_ylabel('Magnitude (dB)', fontsize=12)
        ax2.set_title('📈 Frequency Spectrum (FFT)', fontsize=14, fontweight='bold', pad=15)
        ax2.set_xlim([0, 100])  # Show up to Nyquist frequency
        ax2.grid(alpha=0.3, linestyle='--')
        ax2.set_facecolor('#f8f9fa')
        
        # Mark dominant frequencies
        peak_idx = np.argmax(fft_magnitude[1:]) + 1  # Skip DC component
        peak_freq = fft_freq[peak_idx]
        ax2.axvline(peak_freq, color='red', linestyle=':', linewidth=2, alpha=0.6, label=f'Peak: {peak_freq:.1f} Hz')
        ax2.legend(loc='upper right')
        
        plt.tight_layout()
        plt.show()
    else:
        print("❌ Could not determine signal column in geophone data")
else:
    print("⚠️  No geophone data loaded")

### 4.3 Load GPS Trajectory Data

In [ ]:
# Load GPS trajectory data
gps_path = f'{sample_scene}/{sample_scene}/run{sample_run}_gps.parquet'

try:
    if POLARS_AVAILABLE:
        gps_data = pl.read_parquet(gps_path)
        gps_df = gps_data.to_pandas()
    else:
        gps_df = pd.read_parquet(gps_path)
    
    print(f"GPS trajectory data for Run {sample_run}:")
    print(f"Shape: {gps_df.shape}")
    print(f"Columns: {list(gps_df.columns)}")
    print("\n")
    print(gps_df.head())
    
    # Visualize trajectory if lat/lon columns exist
    if 'latitude' in gps_df.columns and 'longitude' in gps_df.columns:
        plt.figure(figsize=(12, 8))
        plt.plot(gps_df['longitude'], gps_df['latitude'], 
                marker='o', markersize=3, linewidth=1, alpha=0.6)
        plt.scatter(gps_df['longitude'].iloc[0], gps_df['latitude'].iloc[0], 
                   c='green', s=200, marker='o', label='Start', zorder=5)
        plt.scatter(gps_df['longitude'].iloc[-1], gps_df['latitude'].iloc[-1], 
                   c='red', s=200, marker='s', label='End', zorder=5)
        
        # Overlay sensor locations if available
        if 'sensor_loc_df' in locals():
            plt.scatter(sensor_loc_df['longitude'], sensor_loc_df['latitude'], 
                       s=150, c='orange', marker='^', label='Sensors', 
                       edgecolors='black', linewidths=2, zorder=10)
        
        plt.xlabel('Longitude')
        plt.ylabel('Latitude')
        plt.title(f'Vehicle GPS Trajectory - Run {sample_run}', fontweight='bold')
        plt.legend()
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()
    
except FileNotFoundError:
    print(f"File not found: {gps_path}")
except Exception as e:
    print(f"Error loading GPS data: {e}")

## 5. Cross-Scene Comparison

Let's compare characteristics across all six scenes.

In [ ]:
# Load run metadata from all scenes
all_runs = []

for scene in scenes:
    try:
        run_path = f'{scene}/{scene}/run_ids.parquet'
        if POLARS_AVAILABLE:
            runs = pl.read_parquet(run_path).to_pandas()
        else:
            runs = pd.read_parquet(run_path)
        
        # Convert array-like columns to scalar values if needed
        for col in runs.columns:
            if runs[col].dtype == object:
                # Check if first non-null value is an array
                first_val = runs[col].dropna().iloc[0] if len(runs[col].dropna()) > 0 else None
                if first_val is not None and isinstance(first_val, (list, np.ndarray)):
                    # Convert arrays to strings or extract first element
                    runs[col] = runs[col].apply(lambda x: x[0] if isinstance(x, (list, np.ndarray)) and len(x) > 0 else x)
        
        runs['scene'] = scene
        all_runs.append(runs)
        print(f"Loaded {len(runs)} runs from scene {scene}")
    except Exception as e:
        print(f"Error loading {scene}: {e}")

if all_runs:
    all_runs_df = pd.concat(all_runs, ignore_index=True)
    
    # Calculate duration in minutes if length column exists
    if 'length' in all_runs_df.columns:
        all_runs_df['duration_minutes'] = pd.to_timedelta(all_runs_df['length']).dt.total_seconds() / 60
    
    print(f"\n{'='*70}")
    print(f"📊 Cross-Scene Summary")
    print(f"{'='*70}")
    print(f"Total runs across all scenes: {len(all_runs_df)}")
    print(f"Total unique vehicles: {all_runs_df['label'].nunique()}")
    print(f"\nVehicle distribution:")
    vehicle_counts = all_runs_df['label'].value_counts()
    for vehicle, count in vehicle_counts.items():
        print(f"  {vehicle}: {count} runs ({count/len(all_runs_df)*100:.1f}%)")
else:
    print("❌ No run data loaded")

In [ ]:
# Visualize cross-scene comparisons
if 'all_runs_df' in locals():
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Vehicle distribution by scene
    ax1 = axes[0, 0]
    vehicle_scene_counts = pd.crosstab(all_runs_df['scene'], all_runs_df['label'])
    vehicle_scene_counts.plot(kind='bar', ax=ax1, colormap='Set2')
    ax1.set_title('Vehicle Distribution by Scene', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Number of Runs')
    ax1.set_xlabel('Scene')
    ax1.legend(title='Vehicle')
    ax1.grid(alpha=0.3)
    
    # Train/Test split by scene
    ax2 = axes[0, 1]
    split_scene_counts = pd.crosstab(all_runs_df['scene'], all_runs_df['set'])
    split_scene_counts.plot(kind='bar', ax=ax2, stacked=True, 
                            color=['#3498db', '#e74c3c'])
    ax2.set_title('Train/Test Split by Scene', fontsize=14, fontweight='bold')
    ax2.set_ylabel('Number of Runs')
    ax2.set_xlabel('Scene')
    ax2.legend(title='Split', labels=['Train', 'Test'])
    ax2.grid(alpha=0.3)
    
    # Average run duration by vehicle
    if 'duration_minutes' in all_runs_df.columns:
        ax3 = axes[1, 0]
        duration_by_vehicle = all_runs_df.groupby('label')['duration_minutes'].agg(['mean', 'std'])
        duration_by_vehicle['mean'].plot(kind='bar', ax=ax3, color='#3498db', 
                                         yerr=duration_by_vehicle['std'], capsize=5)
        ax3.set_title('Average Run Duration by Vehicle', fontsize=14, fontweight='bold')
        ax3.set_ylabel('Duration (minutes)')
        ax3.set_xlabel('Vehicle')
        ax3.grid(alpha=0.3)
    
    # Total duration by scene
    if 'duration_minutes' in all_runs_df.columns:
        ax4 = axes[1, 1]
        scene_duration = all_runs_df.groupby('scene')['duration_minutes'].sum()
        scene_duration.plot(kind='bar', ax=ax4, color='#2ecc71')
        ax4.set_title('Total Recording Duration by Scene', fontsize=14, fontweight='bold')
        ax4.set_ylabel('Duration (minutes)')
        ax4.set_xlabel('Scene')
        ax4.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 6. Data Quality and Statistical Analysis

In [ ]:
# Check for missing data and compute statistics
if 'all_runs_df' in locals():
    print("=== Data Quality Check ===\n")
    
    # Check for missing values
    print("Missing values in run metadata:")
    print(all_runs_df.isnull().sum())
    print("\n")
    
    # Check for duplicate runs
    duplicates = all_runs_df.duplicated(subset=['scene', 'run_id']).sum()
    print(f"Duplicate runs: {duplicates}\n")
    
    # Summary statistics
    print("=== Summary Statistics ===\n")
    print(f"Total scenes: {all_runs_df['scene'].nunique()}")
    print(f"Total runs: {len(all_runs_df)}")
    print(f"Total vehicles: {all_runs_df['label'].nunique()}")
    print(f"Total train runs: {(all_runs_df['set'] == 'train').sum()}")
    print(f"Total test runs: {(all_runs_df['set'] == 'test').sum()}")
    
    if 'duration_minutes' in all_runs_df.columns:
        print(f"\nTotal recording duration: {all_runs_df['duration_minutes'].sum():.2f} minutes")
        print(f"                         = {all_runs_df['duration_minutes'].sum() / 60:.2f} hours")
        print(f"Average run duration: {all_runs_df['duration_minutes'].mean():.2f} minutes")
        print(f"Shortest run: {all_runs_df['duration_minutes'].min():.2f} minutes")
        print(f"Longest run: {all_runs_df['duration_minutes'].max():.2f} minutes")
    
    # Train/test balance
    print("\n=== Train/Test Balance ===")
    train_test_balance = all_runs_df.groupby('label')['set'].value_counts().unstack(fill_value=0)
    print(train_test_balance)
    
    if 'duration_minutes' in all_runs_df.columns:
        print("\n=== Duration by Vehicle (minutes) ===")
        duration_stats = all_runs_df.groupby('label')['duration_minutes'].agg(['count', 'sum', 'mean', 'std', 'min', 'max'])
        print(duration_stats)

### 6.1 Signal Statistics

Let's compute basic statistics for the sensor signals.

In [ ]:
# Compute statistics for loaded sensor data
signal_stats = {}

if 'mic_df' in locals() and not mic_df.empty:
    # Find signal column
    signal_col = None
    for col in ['value', 'signal', 'amplitude', 'data', 'mic']:
        if col in mic_df.columns:
            signal_col = col
            break
    if signal_col is None:
        numeric_cols = mic_df.select_dtypes(include=[np.number]).columns
        if len(numeric_cols) > 0:
            signal_col = numeric_cols[0]
    
    if signal_col:
        mic_signal = mic_df[signal_col]
        signal_stats['Microphone'] = {
            'Mean': mic_signal.mean(),
            'Std Dev': mic_signal.std(),
            'Min': mic_signal.min(),
            'Max': mic_signal.max(),
            'Median': mic_signal.median(),
            'RMS': np.sqrt(np.mean(mic_signal**2))
        }

if 'geo_df' in locals() and not geo_df.empty:
    # Find signal column
    signal_col = None
    for col in ['value', 'signal', 'amplitude', 'data', 'geo', 'x', 'y', 'z']:
        if col in geo_df.columns:
            signal_col = col
            break
    if signal_col is None:
        numeric_cols = geo_df.select_dtypes(include=[np.number]).columns
        if len(numeric_cols) > 0:
            signal_col = numeric_cols[0]
    
    if signal_col:
        geo_signal = geo_df[signal_col]
        signal_stats['Geophone'] = {
            'Mean': geo_signal.mean(),
            'Std Dev': geo_signal.std(),
            'Min': geo_signal.min(),
            'Max': geo_signal.max(),
            'Median': geo_signal.median(),
            'RMS': np.sqrt(np.mean(geo_signal**2))
        }

if signal_stats:
    stats_df = pd.DataFrame(signal_stats).T
    print("Signal Statistics Summary:")
    print(stats_df)
    
    # Visualize statistics
    fig, ax = plt.subplots(figsize=(12, 6))
    stats_df[['Mean', 'Std Dev', 'RMS']].plot(kind='bar', ax=ax)
    ax.set_title('Signal Statistics Comparison', fontsize=14, fontweight='bold')
    ax.set_ylabel('Value')
    ax.set_xlabel('Sensor Type')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No sensor data available for statistics")

## 7. Key Findings and Summary

### Dataset Characteristics

**M3N-VC Dataset Summary:**

#### 1. **Multi-Scene Coverage**
- 6 unique outdoor environments with varying terrain types
- Diverse weather conditions: Sunny, Rainy, and Windy
- Different road surfaces: Asphalt, Concrete, Dirt & Gravel

#### 2. **Multi-Modal Sensing**
- **Microphone**: 1.6 kHz sampling rate for acoustic signals
- **Geophone**: 200 Hz sampling rate for ground vibrations
- **GPS**: 1 Hz for vehicle trajectory tracking
- Multiple spatially distributed sensor nodes (6-8 per scene)

#### 3. **Vehicle Classification**
- 4 different vehicle types:
  - Mazda CX-30 (cx30)
  - Mercedes-Benz GLE 350 (gle350)
  - Ford Mustang (mustang)
  - Mazda MX-5 (miata)

#### 4. **Data Organization**
- Total duration: ~18 hours of synchronized recordings
- Train/Test split provided for each scene
- Time-synchronized sensor nodes using GPS
- Parquet format for efficient storage and access

#### 5. **Applications**
- Vehicle classification in distributed sensor networks
- Multi-modal signal fusion
- Domain adaptation across different environments
- IoT foundation model training
- Test-time adaptation research

---

### Next Steps

This exploration provides a foundation for:
1. **Feature Engineering**: Extract relevant features from time-series data
2. **Model Development**: Build classification models using multi-modal inputs
3. **Cross-Scene Analysis**: Study domain shift across different environments
4. **Signal Processing**: Apply advanced techniques (spectrograms, wavelets, etc.)
5. **Multi-Node Fusion**: Combine information from multiple sensor nodes

In [ ]:
# Create a comprehensive summary visualization
if 'summary_df' in locals():
    fig = plt.figure(figsize=(16, 10))
    gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)
    
    # Scene overview
    ax1 = fig.add_subplot(gs[0, :])
    x = np.arange(len(summary_df))
    width = 0.35
    ax1.bar(x - width/2, summary_df['train_hours'], width, label='Train', color='#3498db')
    ax1.bar(x + width/2, summary_df['test_hours'], width, label='Test', color='#e74c3c')
    ax1.set_xlabel('Scene')
    ax1.set_ylabel('Hours')
    ax1.set_title('M3N-VC Dataset Overview - Recording Duration by Scene', fontsize=16, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(summary_df['scene'])
    ax1.legend()
    ax1.grid(alpha=0.3, axis='y')
    
    # Terrain types
    ax2 = fig.add_subplot(gs[1, 0])
    terrain_counts = summary_df['terrain'].value_counts()
    colors = plt.cm.Set3(range(len(terrain_counts)))
    ax2.pie(terrain_counts.values, labels=terrain_counts.index, autopct='%1.1f%%', 
            colors=colors, startangle=90)
    ax2.set_title('Terrain Types', fontweight='bold')
    
    # Weather conditions
    ax3 = fig.add_subplot(gs[1, 1])
    weather_counts = summary_df['weather'].value_counts()
    colors = plt.cm.Pastel1(range(len(weather_counts)))
    ax3.pie(weather_counts.values, labels=weather_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90)
    ax3.set_title('Weather Conditions', fontweight='bold')
    
    # Node distribution
    ax4 = fig.add_subplot(gs[1, 2])
    node_counts = summary_df['nodes'].value_counts().sort_index()
    ax4.bar(node_counts.index.astype(str), node_counts.values, color='#2ecc71')
    ax4.set_xlabel('Number of Nodes')
    ax4.set_ylabel('Number of Scenes')
    ax4.set_title('Node Distribution', fontweight='bold')
    ax4.grid(alpha=0.3, axis='y')
    
    # Summary statistics
    ax5 = fig.add_subplot(gs[2, :])
    ax5.axis('off')
    
    total_hours = summary_df['total_hours'].sum()
    train_hours = summary_df['train_hours'].sum()
    test_hours = summary_df['test_hours'].sum()
    total_runs = summary_df['runs'].sum()
    avg_run_duration = (total_hours * 60) / total_runs if total_runs > 0 else 0
    
    summary_text = f"""
    DATASET SUMMARY STATISTICS
    
    Total Scenes: {len(summary_df)}
    Total Recording Duration: {total_hours:.2f} hours ({total_hours*60:.0f} minutes)
    Train Duration: {train_hours:.2f} hours ({(train_hours/total_hours)*100:.1f}%)
    Test Duration: {test_hours:.2f} hours ({(test_hours/total_hours)*100:.1f}%)
    
    Total Runs: {total_runs}
    Average Run Duration: {avg_run_duration:.2f} minutes
    Total Sensor Nodes: {summary_df['nodes'].sum()}
    
    Modalities: Microphone (1.6 kHz), Geophone (200 Hz), GPS (1 Hz)
    Vehicle Types: 4 (Mazda CX-30, Mercedes GLE 350, Ford Mustang, Mazda MX-5)
    """
    
    ax5.text(0.5, 0.5, summary_text, ha='center', va='center', fontsize=12,
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3),
             family='monospace')
    
    plt.suptitle('M3N-VC Dataset: Comprehensive Overview', fontsize=18, fontweight='bold', y=0.98)
    plt.show()

print("\n" + "="*80)
print("Data Exploration Complete!")
print("="*80)